# 01 · Mirar los datos
Primer contacto con el dataset: qué archivos hay, cuántas filas tienen y qué representa cada fila (grano).

In [1]:
from pathlib import Path
Path.cwd()

PosixPath('/home/sebastian/Documentos/GitHub/facultad-main/mineria de datos II - miercoles/notebooks')

In [2]:
from pathlib import Path
import json
import pandas as pd

LANDING = Path('../data/datalake/landing')
sorted(p.name for p in LANDING.iterdir())

['billing_monthly.csv',
 'customers_orgs.csv',
 'marketing_touches.csv',
 'nps_surveys.csv',
 'resources.csv',
 'support_tickets.csv',
 'usage_events_stream',
 'users.csv']

In [3]:
CSV = ['customers_orgs', 'users', 'resources', 'support_tickets',
       'marketing_touches', 'nps_surveys', 'billing_monthly']

tablas = {nombre: pd.read_csv(LANDING / f'{nombre}.csv') for nombre in CSV}

for nombre, df in tablas.items():
    print(f'{nombre:20} {df.shape[0]:>6} filas  {df.shape[1]:>3} columnas')

customers_orgs           80 filas   11 columnas
users                   800 filas    7 columnas
resources               400 filas    7 columnas
support_tickets        1000 filas    8 columnas
marketing_touches      1500 filas    7 columnas
nps_surveys              92 filas    4 columnas
billing_monthly         240 filas    8 columnas


In [4]:
for nombre, df in tablas.items():
    print(f'===== {nombre}')
    display(df.head(3))

===== customers_orgs


,org_id,org_name,industry,hq_region,plan_tier,is_enterprise,signup_date,sales_rep,lifecycle_stage,marketing_source,nps_score
0,org_xaji0y6d,Nimbus Labs 0,Education,sa-east,standard,True,2025-05-26,rep_c,churned,partner,NaN
1,org_pbhsahxt,Nova Tech 1,Education,us-east,standard,False,2025-06-16,rep_d,lead,event,-3.0
2,org_hv3a3zmf,Gamma Data 2,Media,eu-central,pro,True,2025-05-12,rep_c,active,organic,4.0


===== users


,user_id,org_id,email,role,active,created_at,last_login
0,user_9tze5r5u,org_3t60rjiw,user_9tze5r5u@example.com,devops,True,2025-07-10,NaN
1,user_qpgb7r3o,org_afeyuhz1,user_qpgb7r3o@example.com,developer,True,2025-07-15,2025-08-10
2,user_cwbfuk9e,org_ykl1cq99,user_cwbfuk9e@example.com,developer,True,2025-06-12,2025-07-26


===== resources


...

,touch_id,org_id,campaign,channel,timestamp,clicked,converted
0,mkt_7322hr5s,org_zbikcidk,security_week,ads,2025-08-26,False,False
1,mkt_osc0whs1,org_5iqvnb4g,webinar_finops,email,2025-07-23,False,False
2,mkt_4eci9rtj,org_okep7y6w,security_week,email,2025-06-21,False,True


===== nps_surveys


,org_id,survey_date,nps_score,comment
0,org_xaji0y6d,2025-08-22,11.0,Complex billing
1,org_hv3a3zmf,2025-06-13,6.0,Complex billing
2,org_hv3a3zmf,2025-06-18,15.0,Love genAI features


===== billing_monthly


,invoice_id,org_id,month,subtotal,credits,taxes,currency,exchange_rate_to_usd
0,inv_nym31sk0,org_xaji0y6d,2025-06-01,1133.34,NaN,238.00,USD,0.97684
1,inv_540sejok,org_xaji0y6d,2025-07-01,619.17,NaN,130.03,USD,1.00331
2,inv_ilp4vn2v,org_xaji0y6d,2025-08-01,540.79,NaN,113.57,ARS,0.00160


In [5]:
archivo = LANDING / 'usage_events_stream' / 'events_part_0000.jsonl'
with open(archivo) as f:
    primera_linea = f.readline()

print(primera_linea)                                  # texto crudo
json.loads(primera_linea)                             # convertido a diccionario

{"event_id": "evt_nwlh5boz2guf", "timestamp": "2025-08-17T01:55:00Z", "org_id": "org_cvs4f8cg", "resource_id": "res_i8kcmm7d", "service": "networking", "region": "sa-east", "metric": "requests", "value": null, "unit": "count", "cost_usd_increment": 0.9561, "schema_version": 2, "carbon_kg": 0}



{'event_id': 'evt_nwlh5boz2guf',
 'timestamp': '2025-08-17T01:55:00Z',
 'org_id': 'org_cvs4f8cg',
 'resource_id': 'res_i8kcmm7d',
 'service': 'networking',
 'region': 'sa-east',
 'metric': 'requests',
 'value': None,
 'unit': 'count',
 'cost_usd_increment': 0.9561,
 'schema_version': 2,
 'carbon_kg': 0}

In [6]:
eventos = pd.read_json(archivo, lines=True)           # lines=True: un JSON por línea

v1 = eventos[eventos.schema_version == 1].iloc[0]
v2 = eventos[eventos.genai_tokens.notna()].iloc[0]

display(pd.DataFrame({'evento v1': v1, 'evento v2': v2}))

,evento v1,evento v2
event_id,evt_xozb632zko0g,evt_v64of7mqdgpy
timestamp,2025-07-16 11:20:00+00:00,2025-08-08 15:19:00+00:00
org_id,org_rixa11dp,org_rixa11dp
resource_id,res_nmk7yyz6,res_a09w1gxh
service,database,genai
region,ap-northeast,us-east
metric,requests,requests
value,NaN,123.0
unit,count,count
cost_usd_increment,5.6521,13.801
